In [1]:
#STEP 0 — IMPORTS & GLOBALS

In [2]:
import duckdb
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict


In [3]:
DB_PATH = "data/market_data.duckdb"
DATA_DIR = Path("data/raw/financials")
FILES = list(DATA_DIR.glob("*_financials.xlsx"))

In [4]:
#STEP 1 — RESET DATABASE (DROP TABLES)

In [5]:
con = duckdb.connect(DB_PATH)

for t in [
    "financial_income_statement",
    "financial_balance_sheet",
    "financial_cashflow",
    "financial_quarterly_results",
    "shareholding_pattern",
    "stocks_master"
]:
    con.execute(f"DROP TABLE IF EXISTS {t}")

con.close()
print("✅ Step 1: Database reset")


✅ Step 1: Database reset


In [6]:
#STEP 2 — CREATE ALL SCHEMAS (EXACT)

In [7]:
con = duckdb.connect(DB_PATH)

con.execute("""
CREATE TABLE stocks_master (
    stock_id INTEGER PRIMARY KEY,
    symbol TEXT,
    company_name TEXT,
    sector TEXT,
    industry TEXT
)
""")


# ---------- INCOME STATEMENT ----------
con.execute("""
CREATE TABLE financial_income_statement (
    stock_id INTEGER,
    period_end DATE,
    revenue DOUBLE,
    ebitda DOUBLE,
    operating_profit DOUBLE,
    interest_expense DOUBLE,
    depreciation DOUBLE,
    ebt DOUBLE,
    tax_expense_percent DOUBLE,
    net_income DOUBLE,
    net_income_adj DOUBLE,
    eps DOUBLE,
    dividend_Payout_ratio DOUBLE,
    PRIMARY KEY (stock_id, period_end)
)
""")

# ---------- BALANCE SHEET ----------
con.execute("""
CREATE TABLE financial_balance_sheet (
    stock_id INTEGER,
    period_end DATE,

    cash_and_equivalents DOUBLE,
    inventories DOUBLE,
    trade_receivables DOUBLE,
    loans_n_advances DOUBLE,
    other_asset_items DOUBLE,

    fixed_assets DOUBLE,
    gross_block DOUBLE,
    accumulated_depreciation DOUBLE,
    cwip DOUBLE,
    investments DOUBLE,
    total_assets DOUBLE,

    deposits DOUBLE,
    borrowings DOUBLE,
    long_term_borrowings DOUBLE,
    short_term_borrowings DOUBLE,
    other_borrowings DOUBLE,
    lease_liab DOUBLE,

    advance_from_customers DOUBLE,
    non_controlling_int DOUBLE,
    trade_payables DOUBLE,
    other_liability_items DOUBLE,

    equity_share_capital DOUBLE,
    equity_reserves DOUBLE,
    total_liabilities_Equity DOUBLE,

    PRIMARY KEY (stock_id, period_end)
)
""")

# ---------- CASH FLOW ----------
con.execute("""
CREATE TABLE financial_cashflow (
    stock_id INTEGER,
    period_end DATE,
    profit_from_operations DOUBLE,
    receivables DOUBLE,
    inventory DOUBLE,
    payables DOUBLE,
    direct_taxes DOUBLE,
    loans_advances DOUBLE,
    operating_investments DOUBLE,
    operating_deposits DOUBLE,
    other_wc_items DOUBLE,
    working_capital_changes DOUBLE,
    cash_from_operating_activity DOUBLE,

    
    fixed_assets_purchased DOUBLE,
    fixed_assets_sold DOUBLE,
    investments_purchased DOUBLE,
    investments_sold DOUBLE,
    interest_received DOUBLE,
    dividends_received DOUBLE,
    invest_in_subsidiaries DOUBLE,
    acquisition_of_companies DOUBLE,
    other_investing_items DOUBLE,
    cash_from_investing_activity DOUBLE,

    
    proceeds_from_shares DOUBLE,
    proceeds_from_borrowings DOUBLE,
    repayment_of_borrowings DOUBLE,
    proceeds_from_debentures DOUBLE,
    redemption_of_debentures DOUBLE,
    interest_paid_fin DOUBLE,
    dividends_paid DOUBLE,
    financial_liabilities DOUBLE,
    share_application_money DOUBLE,
    other_financing_items DOUBLE,
    cash_from_financing_activity DOUBLE,

    
    net_cash_flow DOUBLE,

    
    PRIMARY KEY (stock_id, period_end)
)
""")

# ---------- QUARTERLY ----------
con.execute("""
CREATE TABLE financial_quarterly_results (
    stock_id INTEGER,
    period_end DATE,
    revenue DOUBLE,
    ebitda DOUBLE,
    operating_profit DOUBLE,
    interest_expense DOUBLE,
    depreciation DOUBLE,
    ebt DOUBLE,
    tax_expense_percent DOUBLE,
    net_income DOUBLE,
    net_income_adj DOUBLE,
    eps DOUBLE,
    PRIMARY KEY (stock_id, period_end)
)
""")

# ---------- SHAREHOLDING ----------
con.execute("""
CREATE TABLE shareholding_pattern (
    stock_id INTEGER,
    period_end DATE,
    promoter_holding DOUBLE,
    fii_holding DOUBLE,
    dii_holding DOUBLE,
    government_holding DOUBLE,
    public_holding DOUBLE,

    PRIMARY KEY (stock_id, period_end)
)
""")

con.close()
print("✅ Step 2: Schemas created")


✅ Step 2: Schemas created


In [8]:
#STEP 3 — POPULATE STOCK MASTER

In [9]:
# STEP 3 — POPULATE STOCK MASTER (FROM TICKERS)

import pandas as pd
import duckdb

# Read tickers
tickers = pd.read_excel("data/raw/Tickers.xlsx")
tickers.columns = tickers.columns.str.lower().str.strip()

# Rename columns (type IS THE SOURCE OF TRUTH)
tickers = tickers.rename(columns={
    "ticker": "symbol",
    "name": "company_name",
    "type": "industry"
})

# Deduplicate
tickers = tickers.drop_duplicates("symbol").reset_index(drop=True)

# Assign stock_id
tickers["stock_id"] = range(1, len(tickers) + 1)

# Keep exact schema expected by stocks_master
tickers = tickers[["stock_id", "symbol", "company_name", "sector", "industry"]]

# Load into DuckDB
con = duckdb.connect(DB_PATH)

# Clear and reload (intentional reset)
con.execute("DELETE FROM stocks_master")

con.register("tickers_df", tickers)
con.execute("""
INSERT INTO stocks_master
SELECT * FROM tickers_df
""")

con.close()

print("✅ stocks_master populated from Tickers.xlsx (industry preserved exactly)")


✅ stocks_master populated from Tickers.xlsx (industry preserved exactly)


In [10]:
import duckdb

con = duckdb.connect(DB_PATH)

stock_map = dict(
    con.execute(
        "SELECT UPPER(symbol), stock_id FROM stocks_master"
    ).fetchall()
)

con.close()

print("✅ stock_map built:", len(stock_map))


✅ stock_map built: 498


In [11]:
#STEP 4 — HELPERS (NORMALIZATION, PARSING)

In [12]:
import pandas as pd
import re

def normalize_key(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"\(.*?\)", "", x)
    x = re.sub(r"[^a-z %]", "", x)
    return x.strip()

def clean_number(x):
    if pd.isna(x):
        return None
    x = str(x).replace(",", "").strip()
    if "%" in x:
        return float(x.replace("%", ""))
    try:
        return float(x)
    except:
        return None

def parse_period(col):
    if str(col).upper() == "TTM":
        return None
    try:
        return pd.to_datetime(col).date()
    except:
        return None


In [13]:
#STEP 6 — CANONICAL MAPPINGS (FROM STEP 5)

In [14]:
PL_MAP = {
    "revenue": ["sales", "revenue"],
    "interest_expense": ["interest"],
    "ebitda": ["operating profit"],
    "operating_profit": ["financing profit"],
    "depreciation": ["depreciation"],
    "ebt" : ["profit before tax"],
    "tax_expense_percent": ["tax %"],
    "net_income": ["net profit"],
    "net_income_adj": ["profit for eps"],
    "eps": ["eps in rs"],
    "dividend_Payout_ratio": ["dividend payout %"]
}


BS_MAP = {


    "cash_and_equivalents" : ["cash equivalents","Cash Equivalents"],
    "inventories" : ["inventories"],
    "trade_receivables" : ["trade receivables"],
    "loans_n_advances" : ["loans n advances"],
    "other_asset_items" : ["other asset items"],

    "fixed_assets" : ["fixed assets"],
    "gross_block"  : ["gross block"],
    "accumulated_depreciation" : ["accumulated depreciation"],
    "cwip" :  ["cwip"],
    "investments" : ["investments"],

    "total_assets" : ["total assets"],

    
    "deposits" :["deposits"],
    "borrowings" : ["borrowings","borrowing","Borrowing"],
    "long_term_borrowings": ["long term borrowings"],
    "short_term_borrowings": ["short term borrowings"],
    "other_borrowings": ["other borrowings"],
    "lease_liab": ["lease liabilities"],

    "advance_from_customers" : ["advance from customers"],
    "non_controlling_int" : ["non controlling int"],
    "trade_payables" : ["trade payables"],
    "other_liability_items" : ["other liability items"],
     
    "equity_share_capital" : ["equity share capital","equity capital","Equity Capital"],
    "equity_reserves" : ["reserves","Reserves"],

    "total_liabilities_Equity" : ["total liabilities"]

}


CF_MAP = {

    "profit_from_operations": ["profit from operations"],
    "receivables" : ["receivables"],
    "inventory" : ["inventory"],
    "payables" : ["payables"],
    "direct_taxes" : ["direct taxes"],
    "loans_advances" : ["loans advances"],
    "operating_investments" : ["operating investments","Operating investments"],
    "operating_deposits" : ["operating deposits","Operating Deposits"],
    "other_wc_items" : ["other wc items","Other WC items"],
    "working_capital_changes" : ["working capital changes","Working capital changes"],
    "cash_from_operating_activity": ["cash from operating activity","Cash from Operating Activity"],


    "fixed_assets_purchased": ["fixed assets purchased","Fixed assets purchased"],
    "fixed_assets_sold": ["fixed assets sold","Fixed assets sold"],
    "investments_purchased" : ["investments purchased","Investments purchased"],
    "investments_sold" : ["investments sold","Investments sold"],
    "interest_received": ["interest received","Interest received"],
    "dividends_received": ["dividends received","Dividends received"],
    "invest_in_subsidiaries" : ["invest in subsidiaries","Invest in subsidiaries"],
    "acquisition_of_companies" : ["acquisition of companies","Acquisition of companies"],
    "other_investing_items" : ["other investing items","Other investing items"],
    "cash_from_investing_activity" : ["cash from investing activity","Cash from Investing Activity"],

    "proceeds_from_shares" : ["proceeds from shares","Proceeds from shares"],
    "proceeds_from_borrowings": ["proceeds from borrowings","Proceeds from borrowings"],
    "repayment_of_borrowings": ["repayment of borrowings","Repayment of borrowings"],
    "proceeds_from_debentures" : ["proceeds from debentures","Proceeds from debentures"],
    "redemption_of_debentures" : ["redemption of debentures","Redemption of debentures"],
    "interest_paid_fin": ["interest paid","Interest paid fin","interest paid fin"],
    "dividends_paid": ["dividends paid","Dividends paid"],
    "financial_liabilities" : ["financial liabilities","Financial liabilities"],
    "share_application_money" : ["share application money","Share application money"],
    "other_financing_items" : ["other financing items","Other financing items"],
    
    "cash_from_financing_activity" : ["cash from financing activity","Cash from Financing Activity"],

    "net_cash_flow": ["net cash flow","Net Cash Flow"]
}


QR_MAP = {
    "revenue": ["sales", "revenue"],
    "interest_expense": ["interest"],
    "ebitda": ["operating profit"],
    "operating_profit": ["financing profit"],
    "depreciation": ["depreciation"],
    "ebt" : ["profit before tax"],
    "tax_expense_percent": ["tax %"],
    "net_income": ["net profit"],
    "net_income_adj": ["profit for eps"],
    "eps": ["eps in rs"]

}


SHAREHOLDING_MAP = {
    "promoter_holding": ["promoters"],
    "fii_holding": ["fiis"],
    "dii_holding": ["diis"],
    "government_holding": ["government"],
    "public_holding": ["public"]
}

In [15]:
#STEP 7 — EXTRACTORS

In [16]:
def extract_from_map(df, col, mapping):
    out = {}
    for _, r in df.iterrows():
        key = normalize_key(r.get("line_item", ""))
        val = clean_number(r[col])
        for canon, aliases in mapping.items():
            if canon not in out:
                for a in aliases:
                    if a in key:
                        out[canon] = val
    return out

def extract_shareholding(df, col, SHAREHOLDING_MAP):
    out = {k: None for k in SHAREHOLDING_MAP}
    for _, r in df.iterrows():
        key = normalize_key(r["category"])
        val = clean_number(r[col])
        for canon, aliases in SHAREHOLDING_MAP.items():
            for a in aliases:
                if a in key and out[canon] is None:
                    out[canon] = val
    return out


In [17]:
#8.2 PROFIT & LOSS — RAW INGEST

In [18]:

con = duckdb.connect(DB_PATH)
for f in FILES:
    symbol = f.stem.replace("_financials", "").upper()
    stock_id = stock_map[symbol]

    xl = pd.ExcelFile(f)

    if "Profit & Loss" not in xl.sheet_names:
        continue

    df = xl.parse("Profit & Loss")
    df.columns = df.columns.str.lower().str.strip()

    for col in df.columns:
        period = parse_period(col)
        if period is None:
            continue

        v = extract_from_map(df, col, PL_MAP)

        con.execute("""
        INSERT OR REPLACE INTO financial_income_statement
        (
            stock_id, period_end,
            revenue, interest_expense,ebitda, operating_profit,
            depreciation, ebt ,tax_expense_percent,
            net_income, net_income_adj,
            eps, dividend_Payout_ratio
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            stock_id,
            period,
            v.get("revenue"),
            v.get("interest_expense"),
            v.get("ebitda"),
            v.get("operating_profit"),
            v.get("depreciation"),
            v.get("ebt"),
            v.get("tax_expense_percent"),
            v.get("net_income"),
            v.get("net_income_adj"),
            v.get("eps"),
            v.get("dividend_Payout_ratio")
        ])


    if "Balance Sheet" not in xl.sheet_names:
        continue

    df = xl.parse("Balance Sheet")
    df.columns = df.columns.str.lower().str.strip()

    for col in df.columns:
        period = parse_period(col)
        if period is None:
            continue

        v = extract_from_map(df, col, BS_MAP)

        con.execute("""
        INSERT OR REPLACE INTO financial_balance_sheet
        (
            stock_id, period_end,
            cash_and_equivalents, inventories, trade_receivables,
            loans_n_advances, other_asset_items,
            fixed_assets, gross_block, accumulated_depreciation,
            cwip, investments, total_assets,
            deposits, borrowings,
            long_term_borrowings, short_term_borrowings, other_borrowings,
            lease_liab, advance_from_customers, non_controlling_int,
            trade_payables, other_liability_items,
            equity_share_capital, equity_reserves,
            total_liabilities_Equity
            
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,?)
        """, [
            stock_id, period,
            
            v.get("cash_and_equivalents"),
            v.get("inventories"),
            v.get("trade_receivables"),
            v.get("loans_n_advances"),
            v.get("other_asset_items"),
            
            v.get("fixed_assets"),
            v.get("gross_block"),
            v.get("accumulated_depreciation"),
            v.get("cwip"),
            v.get("investments"),
            v.get("total_assets"),

            v.get("deposits"),
            v.get("borrowings"),
            v.get("long_term_borrowings"),
            v.get("short_term_borrowings"),
            v.get("other_borrowings"),
            v.get("lease_liab"),

            v.get("advance_from_customers"),
            v.get("non_controlling_int"),
            v.get("trade_payables"),
            v.get("other_liability_items"),

            
            v.get("equity_share_capital"),
            v.get("equity_reserves"),
        
            v.get("total_liabilities_Equity")
            
        ])




    if "Cash Flows" not in xl.sheet_names:
        continue

    df = xl.parse("Cash Flows")
    df.columns = df.columns.str.lower().str.strip()

    for col in df.columns:
        period = parse_period(col)
        if period is None:
            continue

        v = extract_from_map(df, col, CF_MAP)

        con.execute("""
        INSERT OR REPLACE INTO financial_cashflow
     
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            stock_id,
            period,
            v.get("profit_from_operations"),
            v.get("receivables"),
            v.get("inventory"),
            v.get("payables"),
            v.get("direct_taxes"),
            v.get("loans_advances"),
            v.get("operating_investments"),
            v.get("operating_deposits"),
            v.get("other_WC_items"),
            v.get("working_capital_changes"),
            v.get("cash_from_operating_activity"),
            
            v.get("fixed_assets_purchased"),
            v.get("fixed_assets_sold"),
            v.get("investments_purchased"),
            v.get("investments_sold"),
            v.get("interest_received"),
            v.get("dividends_received"),
            v.get("invest_in_subsidiaries"),
            v.get("acquisition_of_companies"),
            v.get("other_investing_items"),
            v.get("cash_from_investing_activity"),
            
            v.get("proceeds_from_shares"),
            v.get("proceeds_from_borrowings"),
            v.get("repayment_of_borrowings"),
            v.get("proceeds_from_debentures"),
            v.get("redemption_of_debentures"),
            v.get("interest_paid_fin"),
            v.get("dividends_paid"),
            v.get("financial_liabilities"),
            v.get("share_application_money"),
            v.get("other_financing_items"),
            v.get("cash_from_financing_activity"),
            
            v.get("net_cash_flow")
        ])

    if "Quarterly Results" not in xl.sheet_names:
        continue

    df = xl.parse("Quarterly Results")
    df.columns = df.columns.str.lower().str.strip()

    for col in df.columns:
        period = parse_period(col)
        if period is None:
            continue

        v = extract_from_map(df, col, QR_MAP)

        con.execute("""
        INSERT OR REPLACE INTO financial_quarterly_results
        (
            stock_id, period_end,
            revenue,
            interest_expense,ebitda , operating_profit ,depreciation,ebt, tax_expense_percent,
            net_income, net_income_adj, eps
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            stock_id,
            period,
            v.get("revenue"),
            v.get("interest_expense"),
            v.get("ebitda"),
            v.get("operating_profit"),
            v.get("depreciation"),
            v.get("ebt"),
            v.get("tax_expense_percent"),
            v.get("net_income"),
            v.get("net_income_adj"),
            v.get("eps"),
        ])


    if "Shareholding Pattern" not in xl.sheet_names:
        continue

    df = xl.parse("Shareholding Pattern")
    df.columns = df.columns.str.lower().str.strip()

    for col in df.columns:
        period = parse_period(col)
        if period is None:
            continue

        v = extract_shareholding(df, col, SHAREHOLDING_MAP)

        con.execute("""
        INSERT OR REPLACE INTO shareholding_pattern
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """, [
            stock_id,
            period,
            v.get("promoter_holding"),
            v.get("fii_holding"),
            v.get("dii_holding"),
            v.get("government_holding"),
            v.get("public_holding"),
        ])

con.close()
print("✅ STEP 8 COMPLETE — RAW DATA INGESTED")

✅ STEP 8 COMPLETE — RAW DATA INGESTED


In [19]:
None           # shares_outstanding calculated later

In [20]:
import duckdb

con = duckdb.connect("data/market_data.duckdb")

con.execute("SHOW TABLES").df()


,name
0,financial_balance_sheet
1,financial_cashflow
2,financial_income_statement
3,financial_quarterly_results
4,shareholding_pattern
5,stocks_master


In [21]:
con.execute("""
SELECT *
FROM stocks_master
LIMIT 20
""").df()

,stock_id,symbol,company_name,sector,industry
0,1,HDFCBANK,HDFC Bank Ltd,Financials,Financials
1,2,ICICIBANK,ICICI Bank Ltd,Financials,Financials
2,3,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
3,4,INFY,Infosys Ltd,Information Technology,Non Financials
4,5,BHARTIARTL,Bharti Airtel Ltd,Communication Services,Non Financials
5,6,LT,Larsen and Toubro Ltd,Industrials,Non Financials
6,7,ITC,ITC Ltd,Consumer Staples,Non Financials
7,8,AXISBANK,Axis Bank Ltd,Financials,Financials
8,9,TCS,Tata Consultancy Services Ltd,Information Technology,Non Financials
9,10,SBIN,State Bank of India,Financials,Financials


In [22]:
con.execute("""
SELECT *
FROM financial_balance_sheet
LIMIT 20
""").df()


,stock_id,period_end,cash_and_equivalents,inventories,trade_receivables,loans_n_advances,other_asset_items,fixed_assets,gross_block,accumulated_depreciation,...,short_term_borrowings,other_borrowings,lease_liab,advance_from_customers,non_controlling_int,trade_payables,other_liability_items,equity_share_capital,equity_reserves,total_liabilities_Equity
0,156,2018-03-01,745.0,0.0,195.0,13.0,7450.0,31.0,43.00,13.00,...,4547.0,0.0,0.0,NaN,0.0,125.0,613.0,16.0,1847.0,9567.0
1,156,2019-03-01,277.0,20.0,296.0,58.0,5549.0,337.0,374.00,38.00,...,581.0,0.0,0.0,NaN,0.0,51.0,698.0,17.0,2894.0,9763.0
2,156,2020-03-01,1179.0,0.0,243.0,154.0,4324.0,608.0,669.00,61.00,...,0.0,8838.0,0.0,NaN,0.0,69.0,1122.0,17.0,2974.0,13021.0
3,156,2021-03-01,838.0,0.0,226.0,211.0,4111.0,837.0,926.00,89.00,...,0.0,5077.0,0.0,NaN,0.0,88.0,746.0,18.0,2810.0,8739.0
4,156,2022-03-01,1022.0,0.0,268.0,197.0,4358.0,816.0,935.00,119.00,...,0.0,5808.0,0.0,NaN,0.0,176.0,1727.0,18.0,3006.0,10734.0
5,156,2023-03-01,726.0,0.0,303.0,258.0,5376.0,880.0,1031.00,151.00,...,0.0,6784.0,0.0,NaN,4.0,136.0,1145.0,36.0,3086.0,11191.0
6,156,2024-03-01,638.0,0.0,328.0,331.0,6866.0,940.0,1129.00,189.00,...,0.0,9472.0,0.0,NaN,0.0,161.0,2032.0,36.0,3414.0,15114.0
7,156,2025-03-01,1092.0,0.0,443.0,390.0,8865.0,1281.0,1516.00,235.00,...,0.0,11160.0,0.0,NaN,0.0,203.0,1340.0,39.0,7026.0,19768.0
8,156,2025-09-01,1279.0,0.0,433.0,9439.0,1236.0,3652.0,NaN,NaN,...,0.0,NaN,213.0,NaN,0.0,145.0,1007.0,40.0,9308.0,24265.0
9,330,2014-03-01,60.0,258.0,265.0,-5.0,84.0,418.0,578.24,159.97,...,108.0,6.0,0.0,0.0,NaN,129.0,138.0,11.0,688.0,1088.0


In [23]:
con.execute("""
SELECT *
FROM financial_cashflow
LIMIT 20
""").df()

,stock_id,period_end,profit_from_operations,receivables,inventory,payables,direct_taxes,loans_advances,operating_investments,operating_deposits,...,repayment_of_borrowings,proceeds_from_debentures,redemption_of_debentures,interest_paid_fin,dividends_paid,financial_liabilities,share_application_money,other_financing_items,cash_from_financing_activity,net_cash_flow
0,156,2018-03-01,486.0,NaN,NaN,NaN,-131.0,NaN,NaN,NaN,...,-55573.0,0.0,0.0,-50.0,-78.0,NaN,NaN,-0.0,1263.0,-437.0
1,156,2019-03-01,495.0,NaN,NaN,NaN,-153.0,NaN,NaN,NaN,...,-30929.0,2902.0,-634.0,-15.0,-85.0,NaN,NaN,-11.0,-80.0,-362.0
2,156,2020-03-01,497.0,NaN,NaN,NaN,-127.0,NaN,NaN,NaN,...,-569596.0,18986.0,-18201.0,-14.0,-210.0,NaN,NaN,-0.0,2290.0,557.0
3,156,2021-03-01,423.0,NaN,NaN,NaN,-116.0,NaN,NaN,NaN,...,-5607.0,0.0,0.0,-17.0,-613.0,NaN,NaN,-0.0,-4836.0,-245.0
4,156,2022-03-01,597.0,NaN,NaN,NaN,-203.0,NaN,NaN,NaN,...,-1738.0,0.0,0.0,-43.0,-486.0,NaN,NaN,-8.0,251.0,52.0
5,156,2023-03-01,285.0,NaN,NaN,NaN,-234.0,NaN,NaN,NaN,...,-2728.0,0.0,0.0,-78.0,-613.0,NaN,NaN,-3.0,556.0,21.0
6,156,2024-03-01,556.0,NaN,NaN,NaN,-264.0,NaN,NaN,NaN,...,-1603.0,0.0,0.0,-62.0,-590.0,NaN,NaN,0.0,1978.0,-67.0
7,156,2025-03-01,672.0,NaN,NaN,NaN,-306.0,NaN,NaN,NaN,...,-1950.0,0.0,0.0,-110.0,-217.0,NaN,NaN,-37.0,3773.0,297.0
8,330,2014-03-01,147.0,-10.0,-33.0,5.0,-24.0,8.0,NaN,NaN,...,-52.0,NaN,NaN,-14.0,0.0,-7.0,NaN,NaN,-73.0,0.0
9,330,2015-03-01,232.0,2.0,-37.0,27.0,-54.0,4.0,NaN,NaN,...,-108.0,NaN,NaN,-3.0,0.0,-7.0,NaN,NaN,-119.0,54.0


In [24]:
con.execute("""
SELECT *
FROM financial_quarterly_results
LIMIT 20
""").df()

,stock_id,period_end,revenue,ebitda,operating_profit,interest_expense,depreciation,ebt,tax_expense_percent,net_income,net_income_adj,eps
0,156,2022-12-01,517.0,328.0,NaN,106.0,12.0,223.0,23.0,172.0,172.0,4.82
1,156,2023-03-01,482.0,272.0,NaN,108.0,12.0,200.0,22.0,155.0,155.0,4.37
2,156,2023-06-01,573.0,353.0,NaN,125.0,13.0,224.0,18.0,184.0,184.0,5.15
3,156,2023-09-01,550.0,313.0,NaN,146.0,14.0,227.0,18.0,186.0,186.0,5.21
4,156,2023-12-01,630.0,375.0,NaN,167.0,14.0,235.0,18.0,192.0,192.0,5.36
5,156,2024-03-01,791.0,286.0,NaN,207.0,17.0,323.0,25.0,243.0,243.0,6.76
6,156,2024-06-01,849.0,577.0,NaN,214.0,16.0,345.0,29.0,244.0,244.0,6.72
7,156,2024-09-01,865.0,541.0,NaN,232.0,17.0,319.0,23.0,245.0,245.0,6.73
8,156,2024-12-01,780.0,444.0,NaN,222.0,17.0,359.0,23.0,276.0,276.0,7.12
9,156,2025-03-01,821.0,461.0,NaN,218.0,20.0,324.0,23.0,250.0,250.0,6.35


In [25]:
con.execute("""
SELECT *
FROM shareholding_pattern
LIMIT 20
""").df()

,stock_id,period_end,promoter_holding,fii_holding,dii_holding,government_holding,public_holding
0,156,2023-03-01,22.02,64.83,2.22,0.00,10.94
1,156,2023-06-01,21.46,63.93,3.79,1.47,10.82
2,156,2023-09-01,20.84,61.87,6.36,1.12,10.92
3,156,2023-12-01,17.78,62.48,8.87,2.54,10.86
4,156,2024-03-01,17.76,63.22,8.34,2.74,10.70
5,156,2024-06-01,15.79,64.56,8.72,2.78,10.92
6,156,2024-09-01,15.71,65.58,8.46,2.63,10.23
7,156,2024-12-01,14.76,66.16,9.73,2.55,9.36
8,156,2025-03-01,14.20,67.22,8.48,2.43,10.08
9,156,2025-06-01,6.27,68.54,7.87,2.28,17.31


In [26]:
import duckdb

con = duckdb.connect("data/market_data.duckdb")

con.execute("SHOW TABLES").df()

con.execute("""
SELECT
*
FROM  financial_balance_sheet fi
JOIN stocks_master sm USING (stock_id)
WHERE sm.symbol = 'RELIANCE'
ORDER BY fi.period_end
LIMIT 20
""").df()

,stock_id,period_end,cash_and_equivalents,inventories,trade_receivables,loans_n_advances,other_asset_items,fixed_assets,gross_block,accumulated_depreciation,...,non_controlling_int,trade_payables,other_liability_items,equity_share_capital,equity_reserves,total_liabilities_Equity,symbol,company_name,sector,industry
0,3,2014-03-01,37984.0,56720.0,9411.0,0.0,31215.0,141417.0,261019.0,119602.0,...,959.0,60860.0,29576.0,2940.0,195747.0,428843.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
1,3,2015-03-01,12545.0,53248.0,5315.0,0.0,34007.0,156458.0,288866.0,132408.0,...,3038.0,59407.0,55291.0,2943.0,215556.0,504486.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
2,3,2016-03-01,11028.0,46486.0,4465.0,841.0,38555.0,184910.0,335499.0,150589.0,...,3356.0,60296.0,109075.0,2948.0,228608.0,598997.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
3,3,2017-03-01,3023.0,48951.0,8177.0,996.0,39393.0,198526.0,361293.0,162767.0,...,2917.0,76595.0,146106.0,2959.0,260750.0,706802.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
4,3,2018-03-01,4255.0,60837.0,17555.0,2327.0,52530.0,403885.0,581284.0,177399.0,...,3539.0,106861.0,167524.0,5922.0,287584.0,811273.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
5,3,2019-03-01,11081.0,67561.0,30089.0,545.0,74882.0,398374.0,596522.0,198148.0,...,8280.0,108309.0,186215.0,5926.0,381186.0,997630.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
6,3,2020-03-01,30920.0,73903.0,19656.0,669.0,119336.0,532658.0,743778.0,211120.0,...,12181.0,96799.0,249736.0,6339.0,442827.0,1163015.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
7,3,2021-03-01,17397.0,81672.0,19014.0,65.0,169878.0,541258.0,775812.0,234554.0,...,99260.0,108897.0,132774.0,6445.0,693727.0,1320065.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
8,3,2022-03-01,36178.0,107778.0,23640.0,130.0,136328.0,627798.0,883614.0,255816.0,...,109499.0,159330.0,131150.0,6765.0,772720.0,1498622.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials
9,3,2023-03-01,68664.0,140008.0,28448.0,176.0,114469.0,724805.0,1018002.0,293197.0,...,113009.0,147172.0,178165.0,6766.0,709106.0,1605882.0,RELIANCE,Reliance Industries Ltd,Energy,Non Financials


In [27]:
con.close()